In [ ]:
%matplotlib inline

import requests
import pandas
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


import ipywidgets as widgets
from IPython.display import display, clear_output

plt.style.use('ggplot')
url_mockapi = "https://6aa1d1d4ccb3db9689a5cff3.mockapi.io/densidade-ibge-2026"

print("Fazendo requisição HTTP para o endpoint JSON...")

res = requests.get(url_mockapi)

if res.status_code == 200:
    dados_json = res.json()
    df_geral = pandas.DataFrame(dados_json)

    print(f"Total de estados encontrados: {len(df_geral)}")
    print(f"Estados disponíveis: {', '.join(df_geral['UF'].unique())}\n")

    seletor_estado = widgets.Dropdown(
        options = sorted(df_geral["UF"].unique()),
        description = "Estado:",
        style = {"description_width": "initial"}
    )

    pesquisar = widgets.Button(
        description="Pesquisar",
        button_style = "success",
        icon="bar-chart"
    )

    saida = widgets.Output()

    def gerar_analise(b):
        with saida:
            clear_output(wait=True)
            uf_pesquisa = seletor_estado.value
            df_uf = df_geral[df_geral['UF'].str.casefold() == uf_pesquisa.casefold()]

            if not df_uf.empty:
                info = df_uf.iloc[0]
                print(f"\nEstado: {info['UF']}")
                print(f"Área territorial: {info.get('Area-territorial')} km²")
                print(f"IDH (2021): {info.get('IDH-2021')}")
                print(f"Densidade demográfica (2022): {info.get('Densidade-demografica-2022')} hab/km²")

                anos = ['2022', '2023', '2024', '2025']

                populacao_milhoes = [info.get(f'Populacao-estado-{ano}', 0) / 1e6 for ano in anos]

                renda_per_capita = [info.get(f'Renda-per-capta-{ano}', 0) for ano in anos]

                df_serie = pandas.DataFrame({
                    'Ano': anos,
                    'Populacao_Milhoes': populacao_milhoes,
                    'Renda_Per_Capta': renda_per_capita
                })

                print("\nEVOLUÇÃO DEMOGRÁFICA E ECONÔMICA:")
                print(df_serie)

                fig, ax1 = plt.subplots(figsize=(10, 5))

                color = '#1f77b4'
                ax1.set_xlabel('Ano de Referência', fontweight='bold', fontsize=11)
                ax1.set_ylabel('Renda Média Per Capita (R$)', color=color, fontweight='bold', fontsize=11)
                line1 = ax1.plot(df_serie['Ano'], df_serie['Renda_Per_Capta'], color=color, linewidth=3, label='Renda Per Capita (R$)')
                ax1.tick_params(axis='y', labelcolor=color)
                ax1.yaxis.set_major_formatter(ticker.StrMethodFormatter('R$ {x:,.0f}'))

                ax2 = ax1.twinx()
                color = '#e67e22'
                ax2.set_ylabel('População Total (Milhões de Hab.)', color=color, fontweight='bold', fontsize=11)
                line2 = ax2.plot(df_serie['Ano'], df_serie['Populacao_Milhoes'], color=color, linewidth=3, linestyle='--', label='População (Milhões)')
                ax2.tick_params(axis='y', labelcolor=color)

                plt.title(f'Potencial Financeiro e Demográfico - {info["UF"]}\n(Subsídio para Análise de Crédito Regional)', fontsize=13, fontweight='bold', pad=15)

                ax1.grid(axis='y', linestyle='-', linewidth=0.7, color='#e0e0e0')
                ax1.spines['top'].set_visible(False)
                ax2.spines['top'].set_visible(False)

                lines = line1 + line2
                labels = [l.get_label() for l in lines]
                ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=10)

                plt.tight_layout()
                plt.show()

            else:
                print(f"Estado '{uf_pesquisa}' não encontrado")

    pesquisar.on_click(gerar_analise)
    display(seletor_estado, pesquisar, saida)

else:
    print(
        f"Erro ao acessar a API. Código: {res.status_code}"
    )